# DeepLabV3+ Fine-Tuning — **4 Bands (R,G,B,NIR)** from Planet 8-Band (SuperDove SR)

This notebook trains a **DeepLabV3+** on **four bands** extracted from Planet **8‑band** GeoTIFF tiles (size **512×512**):
**R, G, B, NIR**. We use **ImageNet-pretrained** encoders and inflate the first conv 3→4 channels (NIR weights = mean of RGB).

## Dependencies

In [1]:

%pip install -q torch torchvision timm segmentation-models-pytorch rasterio albumentations pandas scikit-image


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 91.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

In [3]:

from pathlib import Path
import random

IMAGES_DIR = Path("/content/drive/My Drive/MS Data Science/Thesis/Training/3 Preparing train data/8_band_outputs/subset_800/images")  # change to your imagery folder
LABELS_DIR = Path("/content/drive/My Drive/MS Data Science/Thesis/Training/3 Preparing train data/8_band_outputs/subset_800/labels")  # change to your label folder (same filenames)


# SuperDove SR band map (1-indexed): 1 Coastal, 2 Blue, 3 Green I, 4 Green, 5 Yellow, 6 Red, 7 Red Edge, 8 NIR
RGBNIR_BANDS = (6, 4, 2, 8)  # (R,G,B,NIR)

OUTPUT_DIR = Path("/content/drive/My Drive/MS Data Science/Thesis/Training/4 Model Training/SegFormer/training_output_ndwi_ft")
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

NUM_SAMPLES = 800
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-3

BACKBONES = ['timm-efficientnet-b0', 'mit_b0', 'mit_b1', 'mit_b2']

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [4]:
from typing import List, Tuple

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} total image/label pairs.")

random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]
print(f"Using {len(sampled_pairs)} pairs after sampling (NUM_SAMPLES={NUM_SAMPLES}).")

n_val = int(len(sampled_pairs) * VAL_RATIO)
val_pairs = sampled_pairs[:n_val]
train_pairs = sampled_pairs[n_val:]

print(f"Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")


Found 800 total image/label pairs.
Using 800 pairs after sampling (NUM_SAMPLES=800).
Train pairs: 640, Val pairs: 160


## Dataset & Dataloaders

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path) -> np.ndarray:
    """
    Read a GeoTIFF as (C, H, W) float32 array.
    Expects PlanetScope SuperDove 8-band surface reflectance chips.
    """
    with rasterio.open(path) as src:
        arr = src.read()  # (C, H, W)
    return arr.astype(np.float32)

def compute_ndwi_from_8band(arr: np.ndarray, green_band: int = 4, nir_band: int = 8) -> np.ndarray:
    """
    Compute NDWI from an 8-band PlanetScope SuperDove tile.

    Parameters
    ----------
    arr : np.ndarray
        Image array of shape (C, H, W). Must contain Green and NIR bands.
    green_band : int
        1-based index of the Green band (SuperDove: 4 = Green).
    nir_band : int
        1-based index of the NIR band (SuperDove: 8 = NIR).

    Returns
    -------
    np.ndarray
        NDWI array of shape (H, W) with values roughly in [-1, 1].
    """
    g = arr[green_band - 1].astype(np.float32)
    nir = arr[nir_band - 1].astype(np.float32)
    denom = (g + nir)
    ndwi = (g - nir) / (denom + 1e-6)
    return ndwi

class LakeTilesRGBNIR_NDWI(Dataset):
    """
    Dataset that:
      * Reads 8-band PlanetScope SuperDove SR tiles.
      * Selects RGB+NIR bands.
      * Computes NDWI from Green and NIR.
      * Stacks them into a 5-channel tensor: [R, G, B, NIR, NDWI].
    """
    def __init__(
        self,
        pairs,
        rgbnir_bands = (6, 4, 2, 8),   # (R, G, B, NIR) as 1-based indices
        green_band: int = 4,
        nir_band: int = 8,
        aug=None,
        normalize: bool = True,
        max_reflectance: float = 10000.0,
    ):
        self.pairs = pairs
        self.rgbnir_bands = rgbnir_bands
        self.green_band = green_band
        self.nir_band = nir_band
        self.aug = aug
        self.normalize = normalize
        self.max_reflectance = max_reflectance

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]

        img_arr = read_geotiff(img_path)      # (C, H, W), expect 8 bands
        with rasterio.open(lbl_path) as src:
            lbl_arr = src.read(1)             # (H, W) single-channel mask

        # Select RGB+NIR (4 channels)
        band_idx = [b - 1 for b in self.rgbnir_bands]
        rgbnir = img_arr[band_idx, ...]       # (4, H, W)

        # Compute NDWI from full 8-band stack
        ndwi = compute_ndwi_from_8band(img_arr, green_band=self.green_band, nir_band=self.nir_band)  # (H, W)

        # Stack into 5 channels: [R, G, B, NIR, NDWI]
        img_stack = np.concatenate([rgbnir, ndwi[None, ...]], axis=0)  # (5, H, W)

        # Basic normalization:
        #   * SR bands typically 0–10000 -> scale to 0–1
        #   * NDWI already ~[-1, 1], leave as-is
        if self.normalize:
            img_stack[:4] = np.clip(img_stack[:4] / self.max_reflectance, 0.0, 1.0)

        image = img_stack.transpose(1, 2, 0)      # (H, W, 5) for Albumentations
        mask = lbl_arr.astype(np.float32)         # (H, W)

        if self.aug is not None:
            augmented = self.aug(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        # Back to CHW for PyTorch
        image = np.transpose(image, (2, 0, 1)).astype(np.float32)  # (5, H, W)

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask).float()

        return image, mask

# --- Augmentations & DataLoaders ---

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
])

val_aug = A.Compose([])

train_ds = LakeTilesRGBNIR_NDWI(train_pairs, rgbnir_bands=RGBNIR_BANDS, aug=None, normalize=True)
val_ds   = LakeTilesRGBNIR_NDWI(val_pairs,   rgbnir_bands=RGBNIR_BANDS, aug=None,   normalize=True)

PIN_MEMORY = False
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)

len(train_ds), len(val_ds)


(640, 160)

## Loss & Metrics

In [6]:
import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps: float = 1e-7):
    """
    Dice loss on logits for binary segmentation.

    logits: (B, 1, H, W) or (B, H, W)
    targets: (B, H, W) with {0,1}
    """
    if logits.dim() == 4:
        probs = torch.sigmoid(logits[:, 0])
    else:
        probs = torch.sigmoid(logits)

    targets = targets.float()
    intersection = (probs * targets).sum(dim=(1, 2))
    union = probs.sum(dim=(1, 2)) + targets.sum(dim=(1, 2)) + eps
    dice = (2 * intersection + eps) / union
    return 1.0 - dice.mean()

def bce_dice_loss(logits, targets):
    """
    Combination of BCE-with-logits and Dice loss.
    """
    if logits.dim() == 4:
        logits_flat = logits[:, 0]
    else:
        logits_flat = logits
    bce = F.binary_cross_entropy_with_logits(logits_flat, targets.float())
    dsc = dice_loss_with_logits(logits, targets)
    return bce + dsc

def compute_metrics_from_logits(logits, targets):
    """
    Compute IoU, precision, recall, F1, accuracy from logits and binary targets.
    """
    if logits.dim() == 4:
        probs = torch.sigmoid(logits[:, 0])
    else:
        probs = torch.sigmoid(logits)

    preds = (probs > 0.5).float()
    t = targets.float()

    tp = (preds * t).sum(dim=(1, 2))
    tn = ((1 - preds) * (1 - t)).sum(dim=(1, 2))
    fp = (preds * (1 - t)).sum(dim=(1, 2))
    fn = ((1 - preds) * t).sum(dim=(1, 2))
    eps = 1e-7

    precision = (tp / (tp + fp + eps)).mean().item()
    recall    = (tp / (tp + fn + eps)).mean().item()
    f1        = (2 * precision * recall / (precision + recall + eps))
    iou       = (tp / (tp + fp + fn + eps)).mean().item()
    accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()

    return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)


## Model — Inflate First Conv 3→4

In [7]:
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from torch import optim
import pandas as pd
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IN_CHANNELS = 5  # [R, G, B, NIR, NDWI]

def make_segformer_rgbnir(backbone: str, num_classes: int = 1, pretrained: bool = True) -> nn.Module:
    """
    Create a Unet++ model that takes 5-channel input:
        [R, G, B, NIR, NDWI]
    and uses ImageNet weights for the encoder when available.

    The first conv layer of the encoder is inflated from 3 -> 5 channels by:
        * copying the original RGB weights to the first 3 channels
        * copying the Green channel weights for the extra 2 channels
    """
    encoder_weights = "imagenet" if pretrained else None

    model = smp.Segformer(
        encoder_name=backbone,
        encoder_weights=encoder_weights,
        in_channels=IN_CHANNELS,
        classes=num_classes,
    )

    # Inflate first conv if using pretrained encoder and the first conv expects 3 channels
    if pretrained:
        first_conv = None
        for m in model.encoder.modules():
            if isinstance(m, nn.Conv2d) and m.in_channels == 3:
                first_conv = m
                break

        if first_conv is not None:
            with torch.no_grad():
                old_weight = first_conv.weight  # (out_c, 3, k, k)
                out_c, _, kh, kw = old_weight.shape
                new_weight = torch.zeros(out_c, IN_CHANNELS, kh, kw, device=old_weight.device)

                # Copy RGB into first three channels
                new_weight[:, :3, :, :] = old_weight

                # For NIR and NDWI channels, reuse the "green" kernel (channel index 1)
                new_weight[:, 3, :, :] = old_weight[:, 1, :, :]
                new_weight[:, 4, :, :] = old_weight[:, 1, :, :]

                first_conv.weight = nn.Parameter(new_weight)
                first_conv.in_channels = IN_CHANNELS

    return model

def run_one_epoch(model, loader, optimizer, device, train: bool = True):
    """
    Run a single training/validation epoch.
    """
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            logits = model(imgs)              # (B, 1, H, W)
            loss = bce_dice_loss(logits, lbls)

            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        metrics = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg:
            agg[k] += metrics[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg:
        agg[k] /= max(1, n_batches)

    torch.cuda.empty_cache()
    return avg_loss, agg


Device: cuda


## Training

In [8]:
results_csv = OUTPUT_DIR / "logs" / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","backbone","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

for backbone in BACKBONES:
    print(f"\n=== Training backbone: {backbone} ===")
    model = make_segformer_rgbnir(backbone, num_classes=1, pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val_iou = -1.0
    best_path = OUTPUT_DIR / "models" / f"{backbone}_best_val_iou.pt"

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")

        train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, device, train=True)
        val_loss,   val_metrics   = run_one_epoch(model, val_loader,   optimizer, device, train=False)

        ts = datetime.utcnow().isoformat()

        print(f"Train — loss={train_loss:.4f}, IoU={train_metrics['iou']:.4f}, "
              f"F1={train_metrics['f1']:.4f}, precision={train_metrics['precision']:.4f}, "
              f"recall={train_metrics['recall']:.4f}, acc={train_metrics['accuracy']:.4f}")

        print(f"Val   — loss={val_loss:.4f}, IoU={val_metrics['iou']:.4f}, "
              f"F1={val_metrics['f1']:.4f}, precision={val_metrics['precision']:.4f}, "
              f"recall={val_metrics['recall']:.4f}, acc={val_metrics['accuracy']:.4f}")

        # Log to CSV
        for split, loss, m in [
            ("train", train_loss, train_metrics),
            ("val",   val_loss,   val_metrics),
        ]:
            pd.DataFrame([{
                "timestamp": ts,
                "backbone": backbone,
                "epoch": epoch,
                "split": split,
                "loss": loss,
                "iou": m["iou"],
                "precision": m["precision"],
                "recall": m["recall"],
                "f1": m["f1"],
                "accuracy": m["accuracy"],
            }]).to_csv(results_csv, mode="a", index=False, header=False)

        # Track best model by training IoU (as in earlier version)
        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save({
                "backbone": backbone,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_iou": best_val_iou,
            }, best_path)

print(f"Training complete. Logs: {results_csv}")
print(f"Models saved under: {OUTPUT_DIR / 'models'}")



=== Training backbone: timm-efficientnet-b0 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]


Epoch 1/10


...

model.safetensors:   0%|          | 0.00/54.7M [00:00<?, ?B/s]


Epoch 1/10
Train — loss=0.7488, IoU=0.3134, F1=0.4304, precision=0.5053, recall=0.4648, acc=0.9757
Val   — loss=0.5998, IoU=0.4189, F1=0.5439, precision=0.5230, recall=0.6525, acc=0.9858

Epoch 2/10
Train — loss=0.5561, IoU=0.4271, F1=0.5582, precision=0.6040, recall=0.5951, acc=0.9868
Val   — loss=0.5359, IoU=0.4329, F1=0.5516, precision=0.6092, recall=0.5646, acc=0.9908

Epoch 3/10
Train — loss=0.5760, IoU=0.4123, F1=0.5462, precision=0.6049, recall=0.5760, acc=0.9867
Val   — loss=0.6163, IoU=0.4025, F1=0.5104, precision=0.6663, recall=0.4594, acc=0.9907

Epoch 4/10
Train — loss=0.5517, IoU=0.4332, F1=0.5635, precision=0.6076, recall=0.5983, acc=0.9871
Val   — loss=0.5245, IoU=0.4381, F1=0.5666, precision=0.6579, recall=0.5495, acc=0.9907

Epoch 5/10
Train — loss=0.5327, IoU=0.4422, F1=0.5760, precision=0.6260, recall=0.6002, acc=0.9887
Val   — loss=0.4916, IoU=0.4714, F1=0.6049, precision=0.6285, recall=0.6562, acc=0.9907

Epoch 6/10
Train — loss=0.4638, IoU=0.4989, F1=0.6296, prec

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]


Epoch 1/10
Train — loss=0.7420, IoU=0.3246, F1=0.4388, precision=0.5193, recall=0.4614, acc=0.9750
Val   — loss=0.7198, IoU=0.3323, F1=0.4382, precision=0.5784, recall=0.4137, acc=0.9880

Epoch 2/10
Train — loss=0.5887, IoU=0.4119, F1=0.5392, precision=0.5992, recall=0.5564, acc=0.9880
Val   — loss=0.6079, IoU=0.3881, F1=0.5008, precision=0.5972, recall=0.4914, acc=0.9898

Epoch 3/10
Train — loss=0.5479, IoU=0.4299, F1=0.5676, precision=0.6106, recall=0.5938, acc=0.9883
Val   — loss=0.5620, IoU=0.4506, F1=0.5844, precision=0.5748, recall=0.6731, acc=0.9874

Epoch 4/10
Train — loss=0.5235, IoU=0.4538, F1=0.5896, precision=0.6432, recall=0.6052, acc=0.9896
Val   — loss=0.5726, IoU=0.4272, F1=0.5647, precision=0.5921, recall=0.6328, acc=0.9899

Epoch 5/10
Train — loss=0.4841, IoU=0.4817, F1=0.6179, precision=0.6561, recall=0.6396, acc=0.9903
Val   — loss=0.5046, IoU=0.4548, F1=0.5792, precision=0.7458, recall=0.5144, acc=0.9920

Epoch 6/10
Train — loss=0.4442, IoU=0.5142, F1=0.6412, prec